# Joint action + value pretrain (one shared L2)

Trains the **action head** (PairHead, single-target contract) and the
**value heads** (action-impact design) **together**, both branching from the
**same L2** of `EntityPretrainModel` — action via L3/L4/PairHead, value via
the PlayerConsolidator. **L0+L1 frozen, L2 and up trainable**
(`freeze_below_l2`).

- **Action labels** = pair cache (`--pair-cache-path`).
- **Value labels** = cross-entity cache (`--cross-cache-path`), augmented with
  the P1 future-level labels by `scripts/add_p1_value_labels.py`.
- **Value heads** predict the **actual future LEVEL** `sᵢ(t+K)` of each of the
  5 relative-to-strongest-enemy P1 signals (ship/production/planet advantage,
  safety, fleet-speed advantage) at horizons **{5,10,15,20,50}** — Tier 1 win
  (BCE, highest weight) > Tier 2 the 5 forward signal heads (Huber) > Tier 3
  low-weight aux (backward-Δ anti-shortcut + rank ListMLE + survives). The PBRS
  shaping diffs are derived from these levels later, at PPO time.
- **Warm-start** the shared backbone + action head from the previous joint run
  (`--warm-start`); the new value heads init fresh.
- **Verbose per-head logging** each epoch (action + value groups).

The shared L2 is built at `n_steps=10`, so it consumes both the T=6 pair
cache and the T=10 cross-entity cache via `step_embed[-T:]`.


## 1. Authenticate + pull bundle from GCS

In [ ]:
from google.colab import auth
auth.authenticate_user()
BUCKET = 'gs://orbit-wars-shipping/entity'
# Action cache (pair): Ebi-only, single player, max_fleets=512. The object
# holds ~120k snapshots (NOT pre-filtered to launch turns); the joint trainer
# trains the action head ONLY on the cache's acted_indices (~24.5k launch
# turns) — this acted subset + the higher launch_weight fixes the over-holding
# seen with the old all-snapshots cache.
PAIR_CACHE_PREFIX = 'pair_cache_ebi_acted'
# Value cache (cross-entity) AUGMENTED with the P1 future-level labels
# (scripts/add_p1_value_labels.py -> *_p1.pt; byte-chunked + uploaded).
# LARGER corpus (cap150 = 450 games, 103,418 snapshots, ~3x the cap50 cache)
# to curb the win-head game-identity MEMORIZATION (only ~150 games before).
# Carries split_stems, so joint_pretrain holds out val GAMES and logs the
# 'win memorization gap' (train vs held-out win_acc) each epoch.
CROSS_CACHE_PREFIX = 'cross_cache_joint_cap150_p1'
# Warm-start base: the previous joint run's best ckpt on GCS. We load its
# shared backbone + action head; the NEW value heads init fresh. Use the
# 80%-vs-physical_v4 run (lw=24) so the strong action head carries forward.
WARM_START_RUN  = 'joint_actval_d256_T10_20260603-091552'
WARM_START_SRC  = f'{BUCKET}/runs/{WARM_START_RUN}/joint_best.pt'
print(f'pulling from {BUCKET}\n  action={PAIR_CACHE_PREFIX}  value={CROSS_CACHE_PREFIX}'
      f'\n  warm-start={WARM_START_RUN}')


In [ ]:
import os, subprocess, time, hashlib, json, concurrent.futures
from pathlib import Path

WORK = Path('/content/orbit-wars'); WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

def _gcs_size(url):
    try:
        out = subprocess.run(['gcloud','storage','objects','describe',url,
                              '--format=value(size)'], check=True,
                             capture_output=True, text=True)
        return int(out.stdout.strip())
    except Exception:
        return None

def _cp(src, dst, force=True):
    dst = Path(dst)
    if dst.exists() and not force:
        return dst.stat().st_size
    if dst.exists():
        dst.unlink()
    print(f'  pulling {src} -> {dst.name} ...', flush=True)
    subprocess.run(['gcloud','storage','cp',src,str(dst)], check=True)
    return dst.stat().st_size

def _sha256(p):
    h = hashlib.sha256()
    with open(p,'rb') as fh:
        for blk in iter(lambda: fh.read(1<<20), b''): h.update(blk)
    return h.hexdigest()

def pull_cache(prefix, dst):
    """Chunked (manifest) or single-object cache pull + assemble."""
    dst = Path(dst)
    man_url = f'{BUCKET}/{prefix}.manifest.json'
    if _gcs_size(man_url) is not None:
        man = json.loads(subprocess.run(['gcloud','storage','cat',man_url],
                         check=True, capture_output=True, text=True).stdout)
        total = int(man.get('total_bytes', 0))
        if dst.exists() and total and dst.stat().st_size == total:
            print(f'  {dst.name}: cached ({total/1024**3:.2f} GB)'); return
        cdir = WORK / f'{prefix}_chunks'; cdir.mkdir(exist_ok=True)
        def _pull(spec):
            cp = cdir / spec['name']
            if not (cp.exists() and cp.stat().st_size == int(spec.get('size_bytes',0))):
                subprocess.run(['gcloud','storage','cp',f"{BUCKET}/{spec['name']}",str(cp)], check=True)
            if spec.get('sha256') and _sha256(cp) != spec['sha256']:
                raise RuntimeError(f"sha256 mismatch {spec['name']}")
            return spec['name'], cp.stat().st_size
        print(f'  {prefix}: {len(man["chunks"])} chunks, {total/1024**3:.2f} GB')
        with concurrent.futures.ThreadPoolExecutor(max_workers=len(man['chunks'])) as pool:
            for nm,sz in pool.map(_pull, man['chunks']):
                print(f'    {nm}  {sz/1024**2:.1f} MB')
        if dst.exists(): dst.unlink()
        with open(dst,'wb') as out:
            for c in man['chunks']:
                with open(cdir/c['name'],'rb') as fh:
                    while True:
                        b = fh.read(1<<22)
                        if not b: break
                        out.write(b)
        print(f'  assembled {dst.name}: {dst.stat().st_size/1024**3:.2f} GB')
        import shutil as _sh; _sh.rmtree(cdir, ignore_errors=True)  # free chunk disk
        print(f'  freed chunk dir {cdir.name}')
        return
    for cand in (f'{prefix}.pt', f'{prefix}'):
        if _gcs_size(f'{BUCKET}/{cand}') is not None:
            sz = _cp(f'{BUCKET}/{cand}', dst, force=not dst.exists())
            print(f'  {dst.name}: {sz/1024**3:.2f} GB (single)'); return
    raise RuntimeError(f'no cache for prefix {prefix} on {BUCKET}')

t0 = time.time()
_cp(f'{BUCKET}/code.tgz', WORK/'code.tgz')
_cp(f'{BUCKET}/weights.tgz', WORK/'weights.tgz')
pull_cache(PAIR_CACHE_PREFIX, WORK/'pair_cache.pt')
pull_cache(CROSS_CACHE_PREFIX, WORK/'cross_entity_cache.pt')
_cp(WARM_START_SRC, WORK/'warm_start.pt')  # previous joint_best.pt (~27 MB)
print(f'pull done in {time.time()-t0:.1f}s')


In [ ]:
# Wipe stale extracted code; leave the big caches alone.
!rm -rf agents scripts ckpts
!find . -maxdepth 1 -name '*.pt' ! -name 'pair_cache.pt' ! -name 'cross_entity_cache.pt' ! -name 'warm_start.pt' -delete
!tar xzf code.tgz
!tar xzf weights.tgz
import sys, importlib, gc
for m in list(sys.modules):
    if m.startswith('agents') or m.startswith('scripts'): del sys.modules[m]
importlib.invalidate_caches(); gc.collect()
!find . -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null || true
print('extracted code.tgz + weights.tgz')


## 2. Verify the joint model wiring (value heads off shared L2)

In [ ]:
import torch, agents
from agents.transformer_v2.pretrain.entity_encoder import EntityPretrainModel
from agents.transformer_v2.pretrain.value_heads import ValuePretrainHeads
from agents.transformer_v2.pretrain import joint_pretrain
assert 'Minimal' in (agents.__doc__ or ''), 'stale agents shim — restart kernel'
m = EntityPretrainModel(d_model=256, n_steps=10, with_consolidator=True, with_value_heads=True)
for a in ('entity','cross','dual_role','joint_role','pair_head','consolidator','value_heads'):
    assert getattr(m, a) is not None, f'missing {a} (stale code.tgz)'
assert isinstance(m.value_heads, ValuePretrainHeads)
rep = m.freeze_below_l2()
assert not any(p.requires_grad for p in m.entity.parameters()), 'L1 must be frozen'
assert all(p.requires_grad for p in m.cross.parameters()), 'L2 must be trainable'
assert all(p.requires_grad for p in m.value_heads.parameters()), 'value heads must train'
assert hasattr(joint_pretrain, 'train_joint')
n_tr = sum(p.numel() for p in m.parameters() if p.requires_grad)
print('joint model OK — value heads on shared L2; L1 frozen, L2+ trainable')
print(f'trainable params (L2+): {n_tr:,}')
for k,v in rep.items(): print(f'  {k:<28s} {v:,}')


## 3. GPU check

In [ ]:
import torch
print('cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(cpu)')


## 4. Stage frozen L0 encoders into run-dir layout

In [ ]:
import shutil
from pathlib import Path
PLANET_RUN_DIR = Path('/content/orbit-wars/ckpts/planet')
FLEET_RUN_DIR  = Path('/content/orbit-wars/ckpts/fleet')
COMET_RUN_DIR  = Path('/content/orbit-wars/ckpts/comet')
for d in (PLANET_RUN_DIR, FLEET_RUN_DIR, COMET_RUN_DIR): d.mkdir(parents=True, exist_ok=True)
shutil.copy('/content/orbit-wars/planet_encoder_best.pt', PLANET_RUN_DIR/'planet_encoder_best.pt')
shutil.copy('/content/orbit-wars/fleet_encoder_best.pt',  FLEET_RUN_DIR /'fleet_encoder_best.pt')
shutil.copy('/content/orbit-wars/comet_past_best.pt',     COMET_RUN_DIR /'comet_past_best.pt')
import torch
pc = torch.load(PLANET_RUN_DIR/'planet_encoder_best.pt', map_location='cpu', weights_only=False)
fc = torch.load(FLEET_RUN_DIR /'fleet_encoder_best.pt',  map_location='cpu', weights_only=False)
cc = torch.load(COMET_RUN_DIR /'comet_past_best.pt',     map_location='cpu', weights_only=False)
assert pc['config']['d_model']==fc['config']['d_model']==cc['config']['d_model']==256
print('L0 staged, all d=256')


## 5. Cache paths (the joint CLI reads these .pt directly)

In [ ]:
PAIR_CACHE_PATH  = '/content/orbit-wars/pair_cache.pt'
CROSS_CACHE_PATH = '/content/orbit-wars/cross_entity_cache.pt'
import os
for p in (PAIR_CACHE_PATH, CROSS_CACHE_PATH):
    assert os.path.exists(p), p
    print(f'{p}  {os.path.getsize(p)/1024**3:.2f} GB')


## 5b. Preflight — verify the augmented cross-cache schema

Fails fast (before the 35-epoch run) if the value cache is missing the P1
future-level label columns or carries out-of-range values — i.e. a stale
cache that wasn't run through `scripts/add_p1_value_labels.py`.

In [ ]:
import torch
from agents.transformer_v2.pretrain.cross_entity import CachedCrossEntitySnapshotDataset
from agents.transformer_v2.pretrain.value_signals import P1_FWD_HORIZONS, N_P1_SIGNALS
_ds = CachedCrossEntitySnapshotDataset(CROSS_CACHE_PATH)
_need = ['p1_now','p1_future','p1_valid','p1_back','valid_back','survives_future']
_miss = [c for c in _need if c not in _ds.columns]
assert not _miss, f'cross cache missing P1 label cols {_miss} — rebuild with scripts/add_p1_value_labels.py'
_s = _ds[0]
for _c in _need:
    assert torch.isfinite(_s[_c]).all(), f'{_c} has non-finite values'
_fut = _s['p1_future']
assert tuple(_fut.shape[-2:]) == (N_P1_SIGNALS, len(P1_FWD_HORIZONS)), _fut.shape
assert float(_fut.min()) >= 0.0 and float(_fut.max()) <= 1.0001, (float(_fut.min()), float(_fut.max()))
_n = _ds.columns['p1_future'].shape[0]
print(f'cross cache schema OK: {_n} snapshots; p1_future {tuple(_ds.columns["p1_future"].shape[1:])}')
print('  fwd_horizons', _ds.config.get('p1_fwd_horizons'),
      '| back_horizons', _ds.config.get('p1_back_horizons'))
print('  forward-valid fraction by horizon:',
      [round(x, 3) for x in _ds.columns['p1_valid'].float().mean(0).tolist()])
del _ds, _s  # release the mmap handle before training


## 6. Train (joint action + value, L2~ unfreeze)

In [ ]:
import time
TS = time.strftime('%Y%m%d-%H%M%S')
RUN_TAG = f'joint_actval_d256_T10_{TS}'
OUT_DIR = f'data/runs/joint/{RUN_TAG}'
# hyperparams
D_MODEL, N_STEPS   = 256, 10
BATCH_SIZE, EPOCHS = 16, 35   # MORE epochs: warm-started backbone + fresh value
                              # heads, so the value side needs time to converge.
LR, WEIGHT_DECAY   = 5e-5, 1e-4  # LOWER lr: we warm-start the shared backbone +
                                 # action head, so fine-tune gently (was 1e-4).
LAUNCH_WEIGHT      = 32.0  # up-weight launch rows vs NOOP rows in the per-source CE.
                           # Measured imbalance on acted turns is ~6.7:1 NOOP:launch;
                           # 32 ~= 4.8x balance -> favors launching harder (16->24->32
                           # ramp vs the diagnosed 98.9%% hold-rate). NOT the old
                           # per-cell-BCE pos_weight=600 (a 1-in-4096 cell imbalance).
                           # Watch act/launch_recall vs act/launch_acc: if recall->1
                           # but launch_acc drops, 32 is over-launching (wrong target).
VALUE_COEF         = 1.0   # value-loss weight relative to action loss
VALUE_DROPOUT      = 0.1   # dropout on the value trunk/heads ONLY (not the action
                           # backbone) — regularizes the win head, which otherwise
                           # memorizes the small set of training games. Watch the
                           # per-epoch HOLDOUT win_acc + 'win memorization gap' log.
NUM_WORKERS        = 2     # parallel DataLoader workers; safe now that the joint
                           # loader re-iterates (not itertools.cycle) — raise if
                           # Colab CPU/disk have headroom.
WARM_START_PATH    = '/content/orbit-wars/warm_start.pt'  # previous joint_best.pt
print('RUN_TAG =', RUN_TAG)


In [ ]:
# Run via the ! shell magic with `-u` (unbuffered) so the per-head logs
# STREAM LIVE into the cell. `subprocess.run` block-buffers the child's
# stdout in Colab (nothing shows until it exits) — the ! magic does not.
# IPython substitutes the $VARS from the cells above.
!python -u -m agents.transformer_v2.pretrain.joint_pretrain \
  --out-dir $OUT_DIR \
  --fleet-run-dir $FLEET_RUN_DIR \
  --planet-run-dir $PLANET_RUN_DIR \
  --comet-run-dir $COMET_RUN_DIR \
  --pair-cache-path $PAIR_CACHE_PATH \
  --cross-cache-path $CROSS_CACHE_PATH \
  --d-model $D_MODEL \
  --n-steps $N_STEPS \
  --batch-size $BATCH_SIZE \
  --epochs $EPOCHS \
  --lr $LR \
  --weight-decay $WEIGHT_DECAY \
  --launch-weight $LAUNCH_WEIGHT \
  --value-coef $VALUE_COEF \
  --value-dropout $VALUE_DROPOUT \
  --warm-start $WARM_START_PATH \
  --num-workers $NUM_WORKERS \
  --device cuda \
  --progress-every 50


## 7. Push the trained run back to GCS

In [ ]:
import subprocess
from pathlib import Path
src = Path(OUT_DIR); assert src.is_dir(), src
subprocess.run(['gcloud','storage','cp','--recursive',str(src),f'{BUCKET}/runs/'], check=True)
print('uploaded to', f'{BUCKET}/runs/{src.name}/')
subprocess.run(['gcloud','storage','ls','--long','--readable-sizes',
                f'{BUCKET}/runs/{src.name}/'], check=False)
